# Spark Setup Check

Run each cell top-to-bottom before the interview to verify your environment is working.
All cells should complete without errors and the final cell should print **Setup complete**.

In [ ]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("setup-check")
    .master("local[*]")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

print(f"Spark version: {spark.version}")

In [ ]:
# Verify the data files are accessible
import os

for path in ["/home/jovyan/data/variants.tsv", "/home/jovyan/data/variants_all.tsv.gz", "/home/jovyan/data/gene_panels.csv"]:
    status = "OK" if os.path.exists(path) else "MISSING"
    print(f"  {status}  {path}")

In [ ]:
# Verify Spark can read the TSV and run a basic aggregation
df = (
    spark.read
    .option("sep", "\t")
    .option("header", "true")
    .csv("/home/jovyan/data/variants.tsv")
)

print(f"Row count : {df.count()}")
print(f"Columns   : {len(df.columns)}")

from pyspark.sql import functions as F
df.groupBy("Chr").count().orderBy("Chr").show(5)

In [ ]:
# Verify the output directory is writable
test_path = "/home/jovyan/output/_setup_check"
df.limit(10).write.mode("overwrite").parquet(test_path)

check = spark.read.parquet(test_path)
assert check.count() == 10, "Parquet round-trip failed"

import shutil
shutil.rmtree(test_path)

print("\nSetup complete — your environment is ready.")